In [1]:
pip install ucimlrepo

  Using cached ucimlrepo-0.0.7-py3-none-any.whl.metadata (5.5 kB)
Using cached ucimlrepo-0.0.7-py3-none-any.whl (8.0 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# This one is taken from the instructions in the iris dataset documentation
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
iris = fetch_ucirepo(id=53) 
  
# data (as pandas dataframes) 
X = iris.data.features 
y = iris.data.targets 
  
# metadata 
print(iris.metadata) 

{'uci_id': 53, 'name': 'Iris', 'repository_url': 'https://archive.ics.uci.edu/dataset/53/iris', 'data_url': 'https://archive.ics.uci.edu/static/public/53/data.csv', 'abstract': 'A small classic dataset from Fisher, 1936. One of the earliest known datasets used for evaluating classification methods.\n', 'area': 'Biology', 'tasks': ['Classification'], 'characteristics': ['Tabular'], 'num_instances': 150, 'num_features': 4, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1936, 'last_updated': 'Tue Sep 12 2023', 'dataset_doi': '10.24432/C56C76', 'creators': ['R. A. Fisher'], 'intro_paper': {'ID': 191, 'type': 'NATIVE', 'title': 'The Iris data set: In search of the source of virginica', 'authors': 'A. Unwin, K. Kleinman', 'venue': 'Significance, 2021', 'year': 2021, 'journal': 'Significance, 2021', 'DOI': '1740-9713.01589', 'URL': 'https://www.semanticscholar.org

In [210]:
print(iris.variables) 

           name     role         type demographic  \
0  sepal length  Feature   Continuous        None   
1   sepal width  Feature   Continuous        None   
2  petal length  Feature   Continuous        None   
3   petal width  Feature   Continuous        None   
4         class   Target  Categorical        None   

                                         description units missing_values  
0                                               None    cm             no  
1                                               None    cm             no  
2                                               None    cm             no  
3                                               None    cm             no  
4  class of iris plant: Iris Setosa, Iris Versico...  None             no  


In [211]:
print(X)

     sepal length  sepal width  petal length  petal width
0             5.1          3.5           1.4          0.2
1             4.9          3.0           1.4          0.2
2             4.7          3.2           1.3          0.2
3             4.6          3.1           1.5          0.2
4             5.0          3.6           1.4          0.2
..            ...          ...           ...          ...
145           6.7          3.0           5.2          2.3
146           6.3          2.5           5.0          1.9
147           6.5          3.0           5.2          2.0
148           6.2          3.4           5.4          2.3
149           5.9          3.0           5.1          1.8

[150 rows x 4 columns]


In [212]:
print(y)

              class
0       Iris-setosa
1       Iris-setosa
2       Iris-setosa
3       Iris-setosa
4       Iris-setosa
..              ...
145  Iris-virginica
146  Iris-virginica
147  Iris-virginica
148  Iris-virginica
149  Iris-virginica

[150 rows x 1 columns]


In [6]:
# Splitting the data without library functions
# Sources of the concept: 
# https://www.geeksforgeeks.org/python/how-to-split-data-into-training-and-testing-in-python-without-sklearn/
# https://www.geeksforgeeks.org/python/stratified-sampling-in-pandas/
import pandas as pd

data = pd.concat([X, y], axis=1)
data = data.sample(frac=1, random_state=14).reset_index(drop=True)  # shuffle the data

train = data.groupby("class", group_keys=False).apply(lambda x: x.sample(frac=0.8, random_state=14))
X_train = train.drop("class", axis=1)
y_train = train["class"]

test = data.drop(train.index)
X_test = test.drop("class", axis=1)
y_test = test["class"]
print(X_train.head(1))
print(y_train.head(1))
print(X_test.head(1))
print(y_test.head(1))

     sepal length  sepal width  petal length  petal width
140           5.2          4.1           1.5          0.1
140    Iris-setosa
Name: class, dtype: object
    sepal length  sepal width  petal length  petal width
16           4.7          3.2           1.6          0.2
16    Iris-setosa
Name: class, dtype: object


C:\Users\user1\AppData\Local\Temp\ipykernel_13332\1429524054.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train = data.groupby("class", group_keys=False).apply(lambda x: x.sample(frac=0.8, random_state=14))


In [308]:
y_train.value_counts()

class
Iris-setosa        40
Iris-versicolor    40
Iris-virginica     40
Name: count, dtype: int64

In [310]:
y_test.value_counts()

class
Iris-setosa        10
Iris-versicolor    10
Iris-virginica     10
Name: count, dtype: int64

This dataset needs normalization since kNN would be run on it. The algorithm relies on distance metric so features with higher values would dominate over the smaller ones without data preprocessing. Z-score normalization seems to be a proper approach because it would preserve the initial distribution of the values in the dataset and at the same time centre them around 0. This way the widely varying values (like the ones for petal length feature which are in the interval [1.4, 6]) would not bias the algorithm with unnecessary noise.

In [7]:
def scale_data(X_train, y_train, X_test, y_test):
    X_np_train = X_train.values   

    mean = X_np_train.mean(axis=0)
    std = X_np_train.std(axis=0)

    X_train_scaled = (X_np_train - mean) / std
    X_train =  pd.DataFrame(X_train_scaled, columns=X_train.columns)  

    X_np_test = X_test.values
    X_test_scaled = (X_np_test - mean) / std
    X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns)
    return X_train, y_train, X_test, y_test

In [8]:
import numpy as np
import pandas as pd
X_train_unscaled = X_train.copy()
y_train_unscaled = y_train.copy()
X_test_unscaled = X_test.copy()
y_test_unscaled = y_test.copy()

print(type(X_train))
X_train, y_train, X_test, y_test = scale_data(X_train, y_train, X_test, y_test)
print(X_train.head(3))
print(X_test.head(3))

<class 'pandas.core.frame.DataFrame'>
   sepal length  sepal width  petal length  petal width
0     -0.783896     2.341905     -1.287647    -1.433871
1     -1.763765    -0.145074     -1.402615    -1.302625
2     -0.906379     0.759282     -1.287647    -1.302625
   sepal length  sepal width  petal length  petal width
0     -1.396314     0.307104     -1.230163    -1.302625
1     -1.028863     0.759282     -1.287647    -1.302625
2     -1.396314     0.307104     -1.402615    -1.302625


In [9]:
def compute_euclidian_distance(vector1, vector2):
    distance = 0
    for xi, xj in zip(vector1, vector2):
       distance += (xi - xj) ** 2 
    return np.sqrt(distance)

In [10]:
def get_most_common_label(tuple_list):
    keys = [tuple[1] for tuple in tuple_list]
    occurrences_counter = {key: 0 for key in keys}
    max_occurrences = 0
    most_common_label = tuple_list[0][1]
    for item in tuple_list:
        occurrences_counter[item[1]] += 1
        if (occurrences_counter[item[1]] > max_occurrences):
            max_occurrences = occurrences_counter[item[1]]
            most_common_label = item[1]
    return most_common_label        

In [11]:
def knn_classifier(X_train, Y_train, x_test_sample, k, exclude_point):
    distPairList = []
    for xi, yi in zip(X_train.values, Y_train.values):
        if (exclude_point and np.array_equal(x_test_sample, xi)):
            continue
        currentDistance = compute_euclidian_distance(xi, x_test_sample)
        distPairList.append((xi, yi, currentDistance))
        
    distPairList.sort(key=lambda x: x[2]) 
    knn_list = distPairList[:k]
    return get_most_common_label(knn_list)   

In [12]:
def compute_accuracy(X_train, y_train, X_set, y_set, k, exclude_point):
    correct_classifications = 0
    total_classifications = 0
    for xi, yi in zip(X_set.values, y_set.values):
        current_prediction = knn_classifier(X_train, y_train, xi, k, exclude_point)
        if current_prediction == yi:
            correct_classifications += 1
        total_classifications += 1
    return correct_classifications/total_classifications * 100

In [13]:
# Compute training set accuracy
k = 11

print("Train Set Accuracy:")
print("Accuracy: {:.2f} %".format(compute_accuracy(X_train, y_train, X_train, y_train, k, True)))

Train Set Accuracy:
Accuracy: 96.67 %


In [33]:
def stratify_folds(X, y, fold_groups):
    data = pd.concat([X, y], axis=1)
    data = data.sample(frac=1, random_state=14).reset_index(drop=True)  # shuffle the data

    folds = []
    for fold_index in range(fold_groups):
        fold = data.groupby("class", group_keys=False).apply(lambda x: x.iloc[fold_index::fold_groups])
        folds.append(fold)
    return folds

In [28]:
def get_accuray_scores(X, y, fold_groups, k):
    folds = stratify_folds(X, y, fold_groups)
    cross_validation_accuracy_scores = []
    
    for i in range(fold_groups):
        test = folds[i]
        train = pd.concat(folds[:i] + folds[i+1:])

        X_train_sample = train.drop("class", axis=1)
        y_train_sample = train["class"]
        X_test_sample = test.drop("class", axis=1)
        y_test_sample = test["class"]    

        X_train_sample, y_train_sample, X_test_sample, y_test_sample = scale_data(X_train_sample, y_train_sample, X_test_sample, y_test_sample)        
        current_accuracy = compute_accuracy(X_train_sample, y_train_sample, X_test_sample, y_test_sample, k, True)
        cross_validation_accuracy_scores.append(current_accuracy)
    return cross_validation_accuracy_scores

In [29]:
def get_cross_validation(fold_groups, k):
    cross_validation_accuracy_scores = get_accuray_scores(X_train_unscaled, y_train_unscaled, fold_groups, k)
    np_scores = np.array(cross_validation_accuracy_scores)
    for i, score in enumerate(cross_validation_accuracy_scores, start=1):
        print("Accuracy Fold {}: {:.2f} %".format(i, score))
    print("Average accuracy: {:.2f} %".format(np_scores.mean()))
    print("Standart deviation: {:.2f} %".format(np_scores.std()))
    

In [32]:
print("Cross-validation results for k =", 11)
get_cross_validation(fold_groups = 10, k = 11)
print()
print("Cross-validation results for k =", 2)
get_cross_validation(fold_groups = 10, k = 3)
print()
print("Cross-validation results for k =", 20)
get_cross_validation(fold_groups = 10, k = 20)

Cross-validation results for k = 11


C:\Users\user1\AppData\Local\Temp\ipykernel_13332\3853022289.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fold = data.groupby("class", group_keys=False).apply(lambda x: x.iloc[fold_index::fold_groups], include_groups = True)
C:\Users\user1\AppData\Local\Temp\ipykernel_13332\3853022289.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fold = data.groupby("class", group_keys=False).apply(lambda x: x.iloc[fold_

Accuracy Fold 1: 91.67 %
Accuracy Fold 2: 100.00 %
Accuracy Fold 3: 100.00 %
Accuracy Fold 4: 91.67 %
Accuracy Fold 5: 91.67 %
Accuracy Fold 6: 100.00 %
Accuracy Fold 7: 100.00 %
Accuracy Fold 8: 100.00 %
Accuracy Fold 9: 100.00 %
Accuracy Fold 10: 91.67 %
Average accuracy: 96.67 %
Standart deviation: 4.08 %

Cross-validation results for k = 2


C:\Users\user1\AppData\Local\Temp\ipykernel_13332\3853022289.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fold = data.groupby("class", group_keys=False).apply(lambda x: x.iloc[fold_index::fold_groups], include_groups = True)
C:\Users\user1\AppData\Local\Temp\ipykernel_13332\3853022289.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fold = data.groupby("class", group_keys=False).apply(lambda x: x.iloc[fold_

Accuracy Fold 1: 100.00 %
Accuracy Fold 2: 100.00 %
Accuracy Fold 3: 91.67 %
Accuracy Fold 4: 91.67 %
Accuracy Fold 5: 91.67 %
Accuracy Fold 6: 100.00 %
Accuracy Fold 7: 100.00 %
Accuracy Fold 8: 100.00 %
Accuracy Fold 9: 100.00 %
Accuracy Fold 10: 91.67 %
Average accuracy: 96.67 %
Standart deviation: 4.08 %

Cross-validation results for k = 20


C:\Users\user1\AppData\Local\Temp\ipykernel_13332\3853022289.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fold = data.groupby("class", group_keys=False).apply(lambda x: x.iloc[fold_index::fold_groups], include_groups = True)
C:\Users\user1\AppData\Local\Temp\ipykernel_13332\3853022289.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fold = data.groupby("class", group_keys=False).apply(lambda x: x.iloc[fold_

Accuracy Fold 1: 91.67 %
Accuracy Fold 2: 100.00 %
Accuracy Fold 3: 83.33 %
Accuracy Fold 4: 91.67 %
Accuracy Fold 5: 100.00 %
Accuracy Fold 6: 91.67 %
Accuracy Fold 7: 100.00 %
Accuracy Fold 8: 100.00 %
Accuracy Fold 9: 100.00 %
Accuracy Fold 10: 91.67 %
Average accuracy: 95.00 %
Standart deviation: 5.53 %


In [24]:
# Compute test set accuracy
k = 11
print("Test Set Accuracy:")
print("Accuracy: {:.2f} %".format(compute_accuracy(X_train, y_train, X_test, y_test, k, False)))

Test Set Accuracy:
Accuracy: 96.67 %
